# Similar Movies Debug Runner

Run the setup cell once, set `tmdb_ids`, then execute the result cell to inspect the standalone similar-movies flow with lane labels.

## Cell 1 - Setup

In [2]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from db.postgres import fetch_movie_cards, pool as postgres_pool
from search_v2.similar_movies import run_similar_movies_for_ids

if postgres_pool._closed:
    await postgres_pool.open()

print(f"Project root: {PROJECT_ROOT}")
print("Postgres pool open")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/michaelkeohane/Documents/movie-finder-rag
Postgres pool open


## Cell 2 - Anchors

## Cell 3 - Results

In [4]:
tmdb_ids = [808]
limit = 25

def year_from_release_ts(release_ts: int | None) -> int | None:
    if release_ts is None:
        return None
    return datetime.fromtimestamp(release_ts, tz=timezone.utc).year


result = await run_similar_movies_for_ids(tmdb_ids, limit=limit)

cards = await fetch_movie_cards([item.movie_id for item in result.ranked])
cards_by_id = {card["movie_id"]: card for card in cards}

rows = []
for rank, item in enumerate(result.ranked, start=1):
    card = cards_by_id.get(item.movie_id, {})
    lane_scores = item.evidence.lane_scores
    rows.append(
        {
            "rank": rank,
            "title": card.get("title", "<missing card>"),
            "year": year_from_release_ts(card.get("release_ts")),
            "tmdb_id": item.movie_id,
            "score": round(item.score, 4),
            "dominant_lane": item.evidence.dominant_lane,
            "lanes": ", ".join(item.evidence.candidate_sources),
            "shape": round(lane_scores.get("shape", 0.0), 4),
            "director": round(lane_scores.get("director", 0.0), 4),
            "franchise": round(lane_scores.get("franchise", 0.0), 4),
            "studio": round(lane_scores.get("studio", 0.0), 4),
            "source": round(lane_scores.get("source", 0.0), 4),
            "quality": round(lane_scores.get("quality", 0.0), 4),
        }
    )

print("anchors:", result.anchor_movie_ids)
print("active_anchor_types:", result.active_anchor_types)
print("lane_weights:", result.debug.normalized_lane_weights)
print("vector_space_weights:", result.debug.vector_space_weights)
display(pd.DataFrame(rows))

anchors: [808]
active_anchor_types: ['standard_shape', 'franchise_dominant', 'studio_lineage', 'source_material']
lane_weights: {'shape': 0.44036697247706424, 'franchise': 0.2752293577981651, 'source': 0.11009174311926605, 'quality': 0.027522935779816512, 'format': 0.03669724770642201, 'themes': 0.11009174311926605, 'cast': 0.0, 'specific_award': 0.0, 'director': 1.0, 'studio': 0.06}
vector_space_weights: {'anchor': 0.09278350515463918, 'plot_events': 0.051546391752577324, 'plot_analysis': 0.2061855670103093, 'viewer_experience': 0.2061855670103093, 'watch_context': 0.15463917525773196, 'narrative_techniques': 0.11340206185567012, 'production': 0.061855670103092786, 'reception': 0.11340206185567012}


,rank,title,year,tmdb_id,score,dominant_lane,lanes,shape,director,franchise,studio,source,quality
0,1,Shrek 2,2004,809,0.9996,shape,"shape, franchise, quality, format, themes, studio",1.0000,0.0,0.7,0.9386,0.0000,0.9469
1,2,Tangled,2010,38757,0.3995,shape,"shape, quality, format, themes",0.5387,0.0,0.0,0.0000,0.0000,0.9444
2,3,Enchanted,2007,4523,0.3726,shape,"shape, quality, format, themes",0.4409,0.0,0.0,0.0000,0.0000,0.9385
3,4,Bee Movie,2007,5559,0.3192,shape,"shape, quality, format, themes, studio",0.3006,0.0,0.0,0.8866,0.0000,0.9036
4,5,Shrek Forever After,2010,10192,0.6764,shape,"shape, franchise, source, quality, format, the...",0.5489,0.0,0.7,0.8426,0.3439,0.9123
5,6,The Lego Movie,2014,137106,0.3124,shape,"shape, quality, format, themes",0.3296,0.0,0.0,0.0000,0.0000,0.9575
6,7,The Stolen Princess,2018,463116,0.3106,shape,"shape, source, quality, format, themes",0.4033,0.0,0.0,0.0000,0.3439,0.7316
7,8,Despicable Me,2010,20352,0.3076,shape,"shape, quality, format, themes",0.4035,0.0,0.0,0.0000,0.0000,0.9453
8,9,Shrek the Third,2007,810,0.6192,shape,"shape, franchise, source, quality, format, the...",0.5153,0.0,0.7,0.8866,0.3439,0.9139
9,10,The Road to El Dorado,2000,10501,0.3067,shape,"shape, quality, format, themes, studio",0.3443,0.0,0.0,0.9784,0.0000,0.8996
